# Omnibus — dwell time as a demand proxy (Model B)

RVV gave us **no passenger counts**. But `dwell_s` (`ts_departure_actual_door − ts_arrival_actual_door`) is how long the doors were open — and door-open time is dominated by boardings/alightings. So dwell is a **proxy for demand** (PLAN.md → Model B). This notebook makes that proxy visible two ways:

1. **Where** demand concentrates — a map of mean dwell per stop.
2. **Crowded-slow vs traffic-slow** — dwell vs delay per stop. A stop that's slow *and* high-dwell is congested with people; slow *but* low-dwell loses its time moving between stops (traffic / signals). Different problems, different fixes — and you can't tell them apart from delay alone.

Caveats: Line-1 overlap dropped, `|delay_arr_s| < 7200`, door-opened only. **Crucially we strip schedule-holding, not just close-door events:** a bus's first/last stop is a terminus layover, and any dwell > 120 s is a timing-point hold where the driver waits to leave *on time* — neither is boarding demand. So: exclude each trip's first/last stop, cap dwell at 120 s, and use the **median** (robust to the few remaining holds). Without this the "demand map" just lights up the depots.

In [ ]:
import polars as pl
import folium
import branca.colormap as cmm
import matplotlib.pyplot as plt
import numpy as np

df = pl.read_parquet("../data/parquet/features.parquet")
clean = df.filter((pl.col("source_window") != "Daten_Linie_1_2024-09_2025-08")
                  & (pl.col("delay_arr_s").abs() < 7200))
base = clean.filter(pl.col("door_opened") & (pl.col("dwell_s") > 0)
                    & pl.col("stop_lat").is_not_null())
# strip terminus layovers (first/last stop of each trip) and timing-point holds (>120s),
# leaving dwell that's plausibly boarding/alighting demand
dwell = (base.with_columns(
            is_term=(pl.col("stop_seq") == pl.col("stop_seq").min().over("trip_id"))
                    | (pl.col("stop_seq") == pl.col("stop_seq").max().over("trip_id")))
         .filter(~pl.col("is_term") & (pl.col("dwell_s") <= 120)))
print(f"boarding-dwell events with coords: {dwell.height:,}  "
      f"(dropped {base.height - dwell.height:,} terminus/hold events)")

## 1 — Demand map: mean dwell per stop

Bigger, redder = longer typical door-open time = more boarding activity. This is the closest thing we have to a ridership heatmap.

In [ ]:
per_stop = (dwell.group_by("stop_name").agg(
                pl.col("dwell_s").median().alias("dwell_med"),
                pl.col("dwell_s").mean().alias("dwell_mean"),
                pl.col("delay_arr_s").median().alias("delay_med"),
                pl.col("stop_lat").first().alias("lat"),
                pl.col("stop_lon").first().alias("lon"),
                pl.len().alias("n"))
            .filter(pl.col("n") > 500)
            .sort("dwell_med", descending=True))

dv = per_stop["dwell_med"].to_numpy()
vmin, vmax = float(np.percentile(dv, 5)), float(np.percentile(dv, 95))
cmap = cmm.LinearColormap(["#2c7fb8", "#fee08b", "#d73027"], vmin=vmin, vmax=vmax)
cmap.caption = "median dwell [s] — proxy for boarding demand (blue = quiet, red = busy)"

m = folium.Map(location=[49.0125, 12.0992], zoom_start=12, tiles="CartoDB positron")
for r in per_stop.iter_rows(named=True):
    folium.CircleMarker(
        [r["lat"], r["lon"]], radius=3 + (r["dwell_med"] - vmin) / (vmax - vmin) * 12,
        color="#333", weight=0.4, fill=True, fill_color=cmap(r["dwell_med"]),
        fill_opacity=0.82,
        tooltip=f"{r['stop_name']} — dwell {r['dwell_med']:.0f}s, n={r['n']:,}",
        popup=folium.Popup(f"<b>{r['stop_name']}</b><br>median dwell {r['dwell_med']:.0f}s"
                           f"<br>median delay {r['delay_med']:.0f}s<br>samples {r['n']:,}", max_width=260),
    ).add_to(m)
cmap.add_to(m)
print("top demand stops:")
print(per_stop.head(8).select("stop_name", "dwell_med", "delay_med", "n"))
m

**How to read it.** The red cluster lands on the obvious generators — **Hauptbahnhof** (median 40 s, far ahead of everything), then **Universität**, **Arnulfsplatz**, **HBF Süd/Arcaden**, **Bismarckplatz**, **TechCampus/OTH**. That's the sanity check that dwell really is tracking ridership, not schedule artefacts. Quiet blue stops on the outer ring are low-boarding request stops. For the pitch this *is* the ridership map RVV said we couldn't have — reconstructed from door timings. (Note Klinikum is *not* a dwell hotspot — its delay problem is congestion getting *to* it, not boarding volume; the next chart separates exactly that.)

## 2 — Crowded-slow vs traffic-slow

Plot every stop as (mean dwell, median delay). The two axes decompose "slow":
- **top-right** = busy *and* late → demand-driven congestion (more service / dwell-time signal priority).
- **top-left** = late but *not* busy → time lost between stops → traffic / signals (corridor fix).
- **bottom** = on time → leave alone.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
x = per_stop["dwell_med"].to_numpy(); y = per_stop["delay_med"].to_numpy()
sizes = per_stop["n"].to_numpy() / 400
sc = ax.scatter(x, y, s=sizes, c=y, cmap="RdYlGn_r", alpha=0.75,
                edgecolor="black", linewidth=0.3)
xm, ym = np.median(x), np.median(y)
ax.axvline(xm, color="#888", ls="--", lw=1); ax.axhline(ym, color="#888", ls="--", lw=1)
ax.text(0.98, 0.97, "traffic-slow\n(late, not busy)", transform=ax.transAxes,
        ha="right", va="top", fontsize=9, color="#444")
ax.text(0.98, 0.03, "demand-driven\n(busy & late)", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=9, color="#444")

# label the extreme stops in each quadrant
busy_late = per_stop.filter((pl.col("dwell_med") > xm) & (pl.col("delay_med") > ym)).sort("delay_med", descending=True).head(6)
late_quiet = per_stop.filter((pl.col("dwell_med") < xm) & (pl.col("delay_med") > ym)).sort("delay_med", descending=True).head(6)
for r in pl.concat([busy_late, late_quiet]).iter_rows(named=True):
    ax.annotate(r["stop_name"], (r["dwell_med"], r["delay_med"]),
                fontsize=7, xytext=(4, 3), textcoords="offset points")
ax.set_xlabel("median dwell [s]  →  more boarding demand")
ax.set_ylabel("median arrival delay [s]  →  later")
ax.set_title("Crowded-slow vs traffic-slow — quadrants point to different interventions")
plt.colorbar(sc, label="median delay [s]"); ax.grid(True, alpha=0.25)
fig.tight_layout(); plt.show()

**How to read it.** Stops in the **top-left** (late but low dwell) are where buses bleed time *driving* — these are the candidates for transit-signal priority or a bus lane, and they map straight onto the σ-hotspot corridor from notebook 04. Stops in the **top-right** (late and high dwell) need capacity, not asphalt — more frequent service or all-door boarding. The decomposition is only possible because we have door-level dwell; it's the analytical core of Model B and a clean "we found *why*, not just *where*" beat for the pitch.